# SaaS subscription Analysis

# 1. Discount Sensitivity Analysis
Objective: To determine the mathematical relationship between price reductions and bottom-line profitability.
A negative correlation indicates that as discounts increase, profit decreases at a measurable rate

In [2]:
import pandas as pd

df = pd.read_csv(r"C:\Users\syedz\Downloads\Cleaned SaaS-Sales-Data1.csv", encoding='latin1')

# Correlation as sensitivity score
discount_sensitivity = df['Discount'].corr(df['Profit'])

print("Discount Sensitivity (Profit vs Discount):", discount_sensitivity)

Discount Sensitivity (Profit vs Discount): -0.21948745637176834


# 2. Break-even Discount Threshold
Objective: To identify the "Pivot Point" where the business model fails. 
By using a rolling average, we smooth out individual transaction noise to find the specific discount percentage where the average profit crosses into the negative (the "Death Zone").

In [3]:
df_sorted = df.sort_values('Discount')

# rolling average to smooth noise
df_sorted['rolling_profit'] = df_sorted['Profit'].rolling(100).mean()

# find where profit crosses zero
break_even = df_sorted[df_sorted['rolling_profit'] < 0]['Discount'].min()

print("Break-even Discount Threshold:", round(break_even, 2))

Break-even Discount Threshold: 0.2


# 3. Loss Contribution (Pareto Analysis)
Objective: To identify if the company's losses are widespread or concentrated within a small group of "High-Drain" customers. This helps prioritize account management interventions for the 20% of clients causing 80% of the financial leakage.

In [4]:
loss_df = df[df['Profit'] < 0]

loss_by_customer = loss_df.groupby('Customer')['Profit'].sum().sort_values()

# cumulative contribution
loss_by_customer = loss_by_customer.reset_index()
loss_by_customer['cum_loss'] = loss_by_customer['Profit'].cumsum()
loss_by_customer['cum_percent'] = loss_by_customer['cum_loss'] / loss_by_customer['Profit'].sum()

print(loss_by_customer.head(10))

           Customer     Profit    cum_loss  cum_percent
0          Allstate -9210.0725  -9210.0725     0.058989
1             Bosch -5751.5955 -14961.6680     0.095827
2       Tyson Foods -4924.4428 -19886.1108     0.127368
3            Itochu -4576.3537 -24462.4645     0.156679
4  Costco Wholesale -4421.9135 -28884.3780     0.185001
5    Morgan Stanley -4092.6675 -32977.0455     0.211214
6        Ford Motor -4019.4448 -36996.4903     0.236958
7             FedEx -3841.7660 -40838.2563     0.261564
8      Nissan Motor -3364.6956 -44202.9519     0.283114
9            Anthem -3161.0482 -47364.0001     0.303360


# 4. Product Profit Stability (Risk Analysis)
Objective: To measure the risk profile of each product. A high standard deviation in profit indicates "Volatility"—meaning the product's performance is unpredictable and highly dependent on aggressive discounting or varying unit costs.

In [5]:
stability = df.groupby('Product')['Profit'].std().sort_values(ascending=False)

print(stability)

Product
Alchemy                       1460.921156
Big Ol Database               1099.070067
ContactMatcher                 295.817440
Marketing Suite                182.034224
OneView                        148.319146
Site Analytics                 113.251277
Data Smasher                   106.155455
FinanceHub                     103.758426
Marketing Suite - Gold          81.923600
SaaS Connector Pack             52.753228
Support                         50.312129
SaaS Connector Pack - Gold      35.354983
ChatBot Plugin                  13.384264
Storage                          5.055053
Name: Profit, dtype: float64
